# 🕒 Clearance Model — *how long until this incident clears?*

Three **CatBoost quantile regressors** (P10 / P50 / P90) predict event clearance
time in minutes. Trained CPU-only on a **chronological 80/20 split** (earliest 80%
train, latest 20% test) so the metrics reflect real forward-looking performance.

Run the cells top to bottom. This **retrains and overwrites** `models/clearance/`
and the `clearance` block of `models/registry.json`.

In [ ]:
# 📦 Setup — locate the backend/ root (folder containing app/ and models/)
import os, sys
from pathlib import Path

root = Path.cwd()
while not (root / "app").is_dir() and root != root.parent:
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))
print("✅ Environment ready")
print(f"   📂 Working dir : {os.getcwd()}")
print(f"   🐍 Python      : {sys.version.split()[0]}")

## 📦 Imports & configuration

In [ ]:
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool

from app.config import settings
from app.engines.predict.clearance import CAT_FEATURES, FEATURES, NUM_FEATURES, QUANTILES
from app.ml import evaluate

TARGET        = "duration_min"
MIN_DURATION  = 1.0
MAX_DURATION  = 2880.0     # 48h cap — drop implausible long-tail / data errors
TEST_FRACTION = 0.20
RANDOM_SEED   = 42
_UNKNOWN      = "unknown"
_MODEL_DIR    = settings.models_dir / "clearance"

CATBOOST_PARAMS = dict(iterations=400, depth=4, learning_rate=0.03,
                       l2_leaf_reg=8, random_seed=RANDOM_SEED, verbose=False)

print("✅ Config loaded")
print(f"   🎯 Target     : {TARGET}")
print(f"   🏷️  Categorical: {list(CAT_FEATURES)}")
print(f"   🔢 Numeric    : {list(NUM_FEATURES)}")
print(f"   📐 Quantiles  : {QUANTILES}")

## 📥 Step 1 — Load the cleaned events

In [ ]:
from app.ml.pipeline import load_clean

print("📥 Loading cleaned events ...")
df = load_clean()
print(f"✅ Loaded {len(df):,} events  ·  {df.shape[1]} columns")
print(f"   📅 {df['start_datetime'].min():%Y-%m-%d}  →  {df['start_datetime'].max():%Y-%m-%d}")
df.head(3)

## 🧹 Step 2 — Filter to the trainable range & build features

In [ ]:
print("🧹 Filtering to 1 ≤ duration ≤ 2880 min with a valid start time ...")
before = len(df)
mask = (
    df[TARGET].notna()
    & (df[TARGET] >= MIN_DURATION)
    & (df[TARGET] <= MAX_DURATION)
    & df["start_datetime"].notna()
)
data = df.loc[mask].copy()

for col in CAT_FEATURES:
    data[col] = data[col].astype("string").fillna(_UNKNOWN).replace("", _UNKNOWN).astype(str)
for col in NUM_FEATURES:
    data[col] = data[col].astype("float").fillna(0).astype(int)
data[TARGET] = data[TARGET].astype(float)

print(f"✅ Kept {len(data):,} of {before:,} rows  ({before - len(data):,} dropped)")
print(f"   ⏱️  duration_min →  min={data[TARGET].min():.0f}   "
      f"median={data[TARGET].median():.0f}   max={data[TARGET].max():.0f}")
data[list(FEATURES) + [TARGET]].head(3)

## ✂️ Step 3 — Chronological 80/20 split (no leakage)

In [ ]:
print("✂️  Sorting by start time and splitting earliest 80% / latest 20% ...")
ordered = data.sort_values("start_datetime", kind="mergesort").reset_index(drop=True)
cut = int(len(ordered) * (1 - TEST_FRACTION))
train_df, test_df = ordered.iloc[:cut], ordered.iloc[cut:]

print(f"   🟦 Train: {len(train_df):>5,} rows  |  "
      f"{train_df['start_datetime'].min():%Y-%m-%d} → {train_df['start_datetime'].max():%Y-%m-%d}")
print(f"   🟧 Test : {len(test_df):>5,} rows  |  "
      f"{test_df['start_datetime'].min():%Y-%m-%d} → {test_df['start_datetime'].max():%Y-%m-%d}")
print(f"   🔒 No leakage: {test_df['start_datetime'].min() >= train_df['start_datetime'].max()}")

## 🌲 Step 4 — Train the three quantile models

In [ ]:
def make_pool(frame):
    return Pool(frame[list(FEATURES)], label=frame[TARGET], cat_features=list(CAT_FEATURES))

train_pool, test_pool = make_pool(train_df), make_pool(test_df)
y_train, y_test = train_df[TARGET].to_numpy(), test_df[TARGET].to_numpy()

print("🌲 Training CatBoost quantile models ...")
_MODEL_DIR.mkdir(parents=True, exist_ok=True)
preds = {}
for q, alpha in QUANTILES.items():
    print(f"   → {q.upper()} (alpha={alpha}) ...", end=" ")
    model = CatBoostRegressor(loss_function=f"Quantile:alpha={alpha}", **CATBOOST_PARAMS)
    model.fit(train_pool)
    model.save_model(str(_MODEL_DIR / f"{q}.cbm"))
    preds[q] = model.predict(test_pool)
    print("✅")
print("🎉 All three models trained & saved to models/clearance/")

## 📊 Step 5 — Evaluate (and sanity-check vs a naive baseline)

In [ ]:
# Enforce non-crossing quantiles before scoring.
p10 = np.maximum(preds["p10"], 1.0)
p50 = np.maximum(p10, preds["p50"])
p90 = np.maximum(p50, preds["p90"])

metrics = {
    "p50_mae": round(evaluate.mae(y_test, p50), 3),
    "p50_rmse": round(evaluate.rmse(y_test, p50), 3),
    "pinball_p10": round(evaluate.pinball_loss(y_test, p10, 0.1), 3),
    "pinball_p50": round(evaluate.pinball_loss(y_test, p50, 0.5), 3),
    "pinball_p90": round(evaluate.pinball_loss(y_test, p90, 0.9), 3),
    "p90_coverage": round(evaluate.coverage(y_test, p90), 3),
    "median_baseline_mae": round(evaluate.median_baseline_mae(y_train, y_test), 3),
}

beats = metrics["p50_mae"] < metrics["median_baseline_mae"]
cov_ok = 0.85 <= metrics["p90_coverage"] <= 0.95
print("📊 Held-out test performance")
print(f"   P50 MAE       : {metrics['p50_mae']:.1f} min")
print(f"   P50 RMSE      : {metrics['p50_rmse']:.1f} min")
print(f"   Baseline MAE  : {metrics['median_baseline_mae']:.1f} min  (always predict the median)")
print(f"   {'✅' if beats else '⚠️ '} model {'beats' if beats else 'does NOT beat'} baseline on MAE")
print(f"   P90 coverage  : {metrics['p90_coverage']:.2f}  {'✅ well-calibrated' if cov_ok else '⚠️ out of [0.85, 0.95]'}")
metrics

## 💾 Step 6 — Persist feature list + registry

In [ ]:
(_MODEL_DIR / "features.json").write_text(json.dumps(list(FEATURES), indent=2))

entry = {
    "model": "clearance",
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "target": TARGET,
    "filter": {"min": MIN_DURATION, "max": MAX_DURATION},
    "split": {"type": "time-based", "test_fraction": TEST_FRACTION},
    "n_train": int(len(train_df)),
    "n_test": int(len(test_df)),
    "features": {"categorical": list(CAT_FEATURES), "numeric": list(NUM_FEATURES)},
    "metrics": metrics,
    "artifacts": {q: f"clearance/{q}.cbm" for q in QUANTILES},
}
registry_path = settings.models_dir / "registry.json"
registry = json.loads(registry_path.read_text()) if registry_path.exists() else {}
registry["clearance"] = entry
registry_path.write_text(json.dumps(registry, indent=2))

print("💾 Saved:")
print("   ✅ models/clearance/{p10,p50,p90}.cbm")
print("   ✅ models/clearance/features.json")
print("   ✅ models/registry.json  (clearance block)")
print("\n🏁 Clearance training complete!")